# Task 1: Comparison of Assists per 90 Minutes Between Forwards and Midfielders

## Analytical Question

Do forwards and midfielders have significantly different average assists per 90 minutes in the FIFA World Cup 2026?

## Data Wrangling

In [17]:
import pandas as pd

# Load the dataset and use the second row as the column headings
df = pd.read_csv("../world_cup_2026_players.csv", header=1)

# Remove rows where all values are missing
df = df.dropna(how="all")

# Keep only the variables needed for this analysis
task1_df = df[['Player', 'Pos', 'Ast.1']].copy()

# Rename the assists per 90 column to make it easier to understand
task1_df = task1_df.rename(columns={'Ast.1': 'Assists_per_90'})

# Keep only forwards (FW) and midfielders (MF)
task1_df = task1_df[
    (task1_df['Pos'] == 'FW') | (task1_df['Pos'] == 'MF')
]

# Display the first five rows of the cleaned dataset
task1_df.head()

,Player,Pos,Assists_per_90
1,Brenden Aaronson,MF,0.0
2,Thelo Aasgaard,MF,0.0
3,Hamza Abdelkarim,FW,0.0
9,Yusuf Abdurisag,FW,0.0
17,Ché Adams,FW,0.0



## Data Preparation and Sampling

In [18]:
# Check the number of forwards and midfielders available
print("Number of players in each position:")
print(task1_df['Pos'].value_counts())

# Check for missing values in assists per 90
print("\nMissing values:")
print(task1_df['Assists_per_90'].isna().sum())

Number of players in each position:
Pos
MF    162
FW     82
Name: count, dtype: int64

Missing values:
0


In [22]:
# Take a random sample of 40 forwards
fw_sample = task1_df[task1_df['Pos'] == 'FW'].sample(n=40, random_state=42)

# Take a random sample of 40 midfielders
mf_sample = task1_df[task1_df['Pos'] == 'MF'].sample(n=40, random_state=42)

# Combine the two samples
sample_df = pd.concat([fw_sample, mf_sample])

# Display the sample size for each position
print("Sample size:")
print(sample_df['Pos'].value_counts())

Sample size:
Pos
FW    40
MF    40
Name: count, dtype: int64


## Descriptive Statistics

In [25]:
# Calculate descriptive statistics for assists per 90 by position
descriptive_stats = sample_df.groupby('Pos')['Assists_per_90'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
)

print(descriptive_stats)

     count     mean  median       std  min   max
Pos                                             
FW      40  0.03200     0.0  0.115386  0.0  0.48
MF      40  0.06775     0.0  0.197646  0.0  1.04


## Hypotheses

**Null Hypothesis (H₀):** The average assists per 90 minutes are equal for forwards and midfielders.

**Alternative Hypothesis (H₁):** The average assists per 90 minutes are different for forwards and midfielders.

**H₀:** μFW = μMF  
**H₁:** μFW ≠ μMF

## Inferential Statistics – 95% Confidence Interval

In [28]:
from scipy import stats
import numpy as np

# Store assists per 90 values for forwards and midfielders
fw = fw_sample['Assists_per_90']
mf = mf_sample['Assists_per_90']

# Calculate the difference between the sample means
mean_difference = fw.mean() - mf.mean()

# Calculate the standard error
standard_error = np.sqrt(
    (fw.std()**2 / len(fw)) +
    (mf.std()**2 / len(mf))
)

# Calculate Welch's degrees of freedom
numerator = (
    (fw.std()**2 / len(fw)) +
    (mf.std()**2 / len(mf))
)**2

denominator = (
    ((fw.std()**2 / len(fw))**2 / (len(fw) - 1)) +
    ((mf.std()**2 / len(mf))**2 / (len(mf) - 1))
)

df_ci = numerator / denominator

# Find the critical t-value for a 95% confidence interval
t_critical = stats.t.ppf(0.975, df=df_ci)

# Calculate the margin of error
margin_error = t_critical * standard_error

# Calculate the 95% confidence interval
lower_ci = mean_difference - margin_error
upper_ci = mean_difference + margin_error

# Display the results
print("Mean difference:", round(mean_difference, 4))
print("95% Confidence Interval:", round(lower_ci, 4), "to", round(upper_ci, 4))

Mean difference: -0.0358
95% Confidence Interval: -0.1081 to 0.0366


## Inferential Statistics – Two-Sample t-Test

In [29]:
# Perform Welch's two-sample t-test
t_statistic, p_value = stats.ttest_ind(
    fw,
    mf,
    equal_var=False
)

# Display the results
print("t-statistic:", round(t_statistic, 4))
print("p-value:", round(p_value, 4))

t-statistic: -0.9879
p-value: 0.327


## Conclusion

The sample included 40 forwards and 40 midfielders. Forwards had an average of 0.032 assists per 90 minutes, while midfielders had an average of 0.0678 assists per 90 minutes.

The 95% confidence interval for the difference in means (forwards − midfielders) was -0.1081 to 0.0366. This interval includes zero.

Welch's two-sample t-test produced a t-statistic of -0.9879 and a p-value of 0.327. Since the p-value is greater than 0.05, the null hypothesis is not rejected.

Therefore, there is not enough evidence to conclude that forwards and midfielders have significantly different average assists per 90 minutes in this sample.